# 08. groupby 집계

## 학습 목표
- `groupby`의 split-apply-combine 개념을 이해한다
- 단일/복수 기준으로 그룹화할 수 있다
- `agg`로 여러 통계를 동시에 구할 수 있다
- `transform`으로 그룹별 값을 원본 크기로 확장할 수 있다

---

## 왜 배워야 하나요?

**"카테고리별/지역별/월별 ○○"** 같은 분석은 실무에서 가장 자주 합니다.
`groupby`는 SQL의 `GROUP BY`와 같은 역할로, **Pandas의 가장 강력한 기능** 중 하나입니다.

> **Split-Apply-Combine**:
> 1. Split: 그룹 기준으로 데이터를 나누고
> 2. Apply: 각 그룹에 함수(합계, 평균 등)를 적용하고
> 3. Combine: 결과를 다시 하나의 테이블로 합친다


In [1]:
import pandas as pd

In [2]:
orders1 = pd.read_csv('./data/orders_q1.csv')
orders2 = pd.read_csv('./data/orders_q2.csv')
orders3 = pd.read_csv('./data/orders_q3.csv')
orders4 = pd.read_csv('./data/orders_q4.csv')
all_orders = pd.concat( [orders1, orders2, orders3, orders4] )

In [4]:
all_orders.to_csv('./data/all_orders.csv' , index= False )

In [5]:
products = pd.read_csv('./data/products.csv')
customers = pd.read_csv('./data/customers.csv')

In [7]:
df = pd.merge(all_orders, products, on='상품코드')

In [8]:
df.head(2)

,주문번호,주문일자,고객ID,상품코드,수량,상품명,카테고리,단가
0,ORD00001,2024-01-02,C019,P001,3,노트북,컴퓨터,1350000
1,ORD00002,2024-01-03,C028,P013,3,이어폰,음향,55000


In [10]:
df['매출'] = df['수량'] * df['단가']

In [12]:
df.to_csv('./data/order_product.csv' , index=False)

---
## 1. 기본 groupby


In [13]:
df.head(3)

,주문번호,주문일자,고객ID,상품코드,수량,상품명,카테고리,단가,매출
0,ORD00001,2024-01-02,C019,P001,3,노트북,컴퓨터,1350000,4050000
1,ORD00002,2024-01-03,C028,P013,3,이어폰,음향,55000,165000
2,ORD00003,2024-01-04,C026,P002,3,무선마우스,주변기기,35000,105000


In [16]:
df['카테고리'].unique()

array(['컴퓨터', '음향', '주변기기', '저장장치', '액세서리', '웨어러블'], dtype=object)

In [17]:
# 카테고리별로 매출 총합을 구하라.
df.groupby('카테고리')['매출'].sum()

카테고리
액세서리      4378000
웨어러블     25920000
음향       17965000
저장장치     12065000
주변기기     29420000
컴퓨터     209680000
Name: 매출, dtype: int64

In [19]:
# 카테고리별로  매출 평균을 구하라.
df.groupby('카테고리')['매출'].mean()

카테고리
액세서리    4.560417e+04
웨어러블    6.646154e+05
음향      1.891053e+05
저장장치    1.945968e+05
주변기기    8.577259e+04
컴퓨터     1.823304e+06
Name: 매출, dtype: float64

In [20]:
# 카테고리별로, 매출의  총합, 평균, 데이터갯수도 구하라.

In [25]:
df.groupby('카테고리')['매출'].agg( [ 'sum', 'mean', 'count', 'max', 'min' ] )

,sum,mean,count,max,min
카테고리,,,,,
액세서리,4378000,4.560417e+04,96,190000,12000
웨어러블,25920000,6.646154e+05,39,1600000,320000
음향,17965000,1.891053e+05,95,600000,55000
저장장치,12065000,1.945968e+05,62,475000,95000
주변기기,29420000,8.577259e+04,343,445000,15000
컴퓨터,209680000,1.823304e+06,115,6750000,450000


---
## 2. 여러 컬럼으로 그룹화


In [28]:
# 지역별, 카테고리별로  매출 총합 구하라.
df = pd.merge(df , customers, on='고객ID')

In [30]:
df.groupby( ['지역','카테고리' ] )['매출'].sum()

지역  카테고리
경기  액세서리     1350000
    웨어러블     8320000
    음향       4560000
    저장장치     3515000
    주변기기    10588000
    컴퓨터     58840000
광주  액세서리      372000
    웨어러블     5120000
    음향        745000
    저장장치     1710000
    주변기기     1391000
    컴퓨터      6080000
대구  액세서리      320000
    웨어러블     2240000
    음향       1465000
    저장장치     1425000
    주변기기     3164000
    컴퓨터     18690000
대전  액세서리      592000
    웨어러블      640000
    음향       1485000
    저장장치     1045000
    주변기기     2477000
    컴퓨터     40050000
부산  액세서리       24000
    음향        480000
    저장장치       95000
    주변기기      788000
    컴퓨터      2250000
서울  액세서리     1608000
    웨어러블     8960000
    음향       8465000
    저장장치     3135000
    주변기기     9323000
    컴퓨터     68670000
인천  액세서리      112000
    웨어러블      640000
    음향        765000
    저장장치     1140000
    주변기기     1689000
    컴퓨터     15100000
Name: 매출, dtype: int64

In [32]:
df.groupby( ['지역','카테고리' ] )['매출'].sum().unstack(fill_value=0)


카테고리,액세서리,웨어러블,음향,저장장치,주변기기,컴퓨터
지역,,,,,,
경기,1350000,8320000,4560000,3515000,10588000,58840000
광주,372000,5120000,745000,1710000,1391000,6080000
대구,320000,2240000,1465000,1425000,3164000,18690000
대전,592000,640000,1485000,1045000,2477000,40050000
부산,24000,0,480000,95000,788000,2250000
서울,1608000,8960000,8465000,3135000,9323000,68670000
인천,112000,640000,765000,1140000,1689000,15100000


---
## 3. 여러 컬럼에 여러 통계


In [33]:
# 카테고리별로,  매출총합, 수량 평균, 주문번호 건수 를 가져와라.

In [35]:
df.groupby('카테고리').agg( {'매출' : 'sum' , '수량' : 'mean', '주문번호': 'count' }  )

,매출,수량,주문번호
카테고리,,,
액세서리,4378000,1.927083,96
웨어러블,25920000,2.076923,39
음향,17965000,1.957895,95
저장장치,12065000,2.048387,62
주변기기,29420000,1.959184,343
컴퓨터,209680000,2.060870,115


---
## 4. `transform` - 그룹 통계를 원본 크기로 확장

집계 결과를 원본 행마다 붙이고 싶을 때 사용합니다.


In [38]:
# 카테고리별로 매출 평균을 구하고,
# 매출에서 매출평균을 빼서,  '평균대비차이'라는 컬럼을 만들자.

# 아래 식은, 카테고리별로만 결과가 나온다.
df.groupby('카테고리')['매출'].mean()

카테고리
액세서리    4.560417e+04
웨어러블    6.646154e+05
음향      1.891053e+05
저장장치    1.945968e+05
주변기기    8.577259e+04
컴퓨터     1.823304e+06
Name: 매출, dtype: float64

In [41]:
# 원본 df 의 갯수만큼 카테고리별 매출을 만드는 방법
df['카테고리별매출평균'] = df.groupby('카테고리')['매출'].transform('mean')

In [46]:
pd.options.display.float_format = '{:.6f}'.format 

In [47]:
df

,주문번호,주문일자,고객ID,상품코드,수량,상품명,카테고리,단가,매출,고객명,성별,연령,지역,가입일,회원등급,카테고리별매출평균
0,ORD00001,2024-01-02,C019,P001,3,노트북,컴퓨터,1350000,4050000,엄정화,여,54,대전,2022-04-01,골드,1823304.347826
1,ORD00002,2024-01-03,C028,P013,3,이어폰,음향,55000,165000,김유정,여,53,서울,2022-05-16,실버,189105.263158
2,ORD00003,2024-01-04,C026,P002,3,무선마우스,주변기기,35000,105000,김민수,여,27,경기,2022-05-06,실버,85772.594752
3,ORD00004,2024-01-05,C037,P004,2,27인치모니터,컴퓨터,450000,900000,고윤정,여,54,경기,2022-06-30,골드,1823304.347826
4,ORD00005,2024-01-06,C026,P014,2,태블릿,컴퓨터,680000,1360000,김민수,여,27,경기,2022-05-06,실버,1823304.347826
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745,ORD00746,2024-12-30,C006,P006,2,웹캠,주변기기,65000,130000,신민아,여,34,경기,2022-01-26,일반,85772.594752
746,ORD00747,2024-12-30,C044,P004,1,27인치모니터,컴퓨터,450000,450000,원빈,남,24,광주,2022-08-04,일반,1823304.347826
747,ORD00748,2024-12-31,C009,P008,3,외장SSD,저장장치,95000,285000,황민현,여,26,서울,2022-02-10,일반,194596.774194
748,ORD00749,2024-12-31,C045,P003,1,기계식키보드,주변기기,89000,89000,이민호,여,45,서울,2022-08-09,골드,85772.594752


In [49]:
df['평균대비차이'] = df['매출'] - df['카테고리별매출평균']

In [52]:
# 매출이 평균보다 높은 데이터만 가져와라.
df[ df['평균대비차이'] > 0  ]

,주문번호,주문일자,고객ID,상품코드,수량,상품명,카테고리,단가,매출,고객명,성별,연령,지역,가입일,회원등급,카테고리별매출평균,평균대비차이
0,ORD00001,2024-01-02,C019,P001,3,노트북,컴퓨터,1350000,4050000,엄정화,여,54,대전,2022-04-01,골드,1823304.347826,2226695.652174
2,ORD00003,2024-01-04,C026,P002,3,무선마우스,주변기기,35000,105000,김민수,여,27,경기,2022-05-06,실버,85772.594752,19227.405248
9,ORD00010,2024-01-08,C001,P003,3,기계식키보드,주변기기,89000,267000,김태희,여,37,경기,2022-01-01,실버,85772.594752,181227.405248
11,ORD00012,2024-01-08,C034,P009,5,노트북거치대,주변기기,45000,225000,이영희,여,59,경기,2022-06-15,일반,85772.594752,139227.405248
18,ORD00019,2024-01-15,C016,P007,2,블루투스스피커,음향,120000,240000,윤태호,여,36,광주,2022-03-17,실버,189105.263158,50894.736842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,ORD00744,2024-12-28,C025,P003,1,기계식키보드,주변기기,89000,89000,김세정,여,28,대전,2022-05-01,일반,85772.594752,3227.405248
745,ORD00746,2024-12-30,C006,P006,2,웹캠,주변기기,65000,130000,신민아,여,34,경기,2022-01-26,일반,85772.594752,44227.405248
747,ORD00748,2024-12-31,C009,P008,3,외장SSD,저장장치,95000,285000,황민현,여,26,서울,2022-02-10,일반,194596.774194,90403.225806
748,ORD00749,2024-12-31,C045,P003,1,기계식키보드,주변기기,89000,89000,이민호,여,45,서울,2022-08-09,골드,85772.594752,3227.405248


---
## 5. 그룹별 상위 N개


In [57]:
# 카테고리별, 상품명별로 매출 총합을 구해주세요.
# 그리고나서 매출 상위 2개를 알려주세요.
df.groupby(['카테고리','상품명'])['매출'].sum().sort_values(ascending=False).head(2)

카테고리  상품명    
컴퓨터   노트북        140400000
      27인치모니터     41400000
Name: 매출, dtype: int64

---
## 실습문제

> 공통 준비 코드:
> ```python
> import glob
> orders = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("../data/orders_q*.csv"))], ignore_index=True)
> products = pd.read_csv("../data/products.csv")
> customers = pd.read_csv("../data/customers.csv")
> df = orders.merge(products, on="상품코드").merge(customers[["고객ID", "지역", "성별"]], on="고객ID")
> df["매출"] = df["수량"] * df["단가"]
> ```

### 문제 1. 카테고리별 매출

카테고리별로 **총 매출**, **평균 매출**, **주문 건수**를 한번에 구하세요.


In [ ]:
# 아래에 코드를 작성하세요


### 문제 2. 지역별 × 카테고리별 매출 (크로스 집계)

지역과 카테고리를 기준으로 매출 합계를 구하고, `unstack()`으로 가로 펼치기를 하세요.


In [ ]:
# 아래에 코드를 작성하세요


### 문제 3. 매출 Top 10 상품

상품명별 총 매출을 구하고 상위 10개를 출력하세요.


In [ ]:
# 아래에 코드를 작성하세요


### 문제 4. 성별 x 카테고리별 구매 패턴

성별과 카테고리를 기준으로 **매출 합계**와 **주문 건수**를 동시에 구하세요.


In [ ]:
# 아래에 코드를 작성하세요
